# Hybrid crawl and benchmark — 100-paper tar.gz batches

Experiment for Dang's first 2,500 IDs. Sources are extracted and filtered to .tex/.bib locally, then stored on Drive as tar.gz level 1 batches of 100 original paper IDs. Metadata, references, version status and empty version directories are preserved. No raw corpus mirror is uploaded. Partial batch snapshots protect in-progress work; only QA-complete batches are final payloads.


In [ ]:
# Edit this cell. Use a NEW run ID for each new experiment.
MODE = "partition"  # sample, partition (partial assigned range), or final
RUN_ID = "dang-0000-2499-tgz100"  # Set a unique run ID matching MEMBER and offsets
MEMBER = "dang"  # Choose: dang, thien, tien, hieu
if MEMBER not in ("dang", "thien", "tien", "hieu"):
    raise ValueError("MEMBER must be dang, thien, tien, or hieu")
RANGES = (MEMBER,)  # final mode: select all four members
OFFSET = 0  # Offset within the selected member range
COUNT = 2500  # Number of IDs within the selected member range
CHUNK_SIZE = 100  # Number of selected IDs per chunk before phase partitioning
WITH_REFERENCES = True
SNAPSHOT_PATH = "/content/arxiv-assigned-20000.jsonl"
SNAPSHOT_DRIVE_PATH = "/content/drive/MyDrive/arxiv-inputs/arxiv-assigned-20000.jsonl"
SNAPSHOT_SHA256 = "033d7bded327d3aa1595f5502ebfc44bcfe353a89a2011b297742d287ba4de15"
CHECKPOINT_S = 600  # Checkpoint target in seconds; not a strict deadline
RESUME = False  # True in a new runtime after a saved run/timeout
DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"
TEAM_ID = "arxiv-nhom1"
INITIALIZE_TEAM_FOLDER = False  # True once, after checking the intended Drive folder
CODE_REF = "experiment/hybrid-archive-batches"  # New runs resolve this branch to a SHA; resume uses the saved SHA
REPO = "https://github.com/DankDaPancake/CodeRepo.git"
ARCHIVE_BATCH_SIZE = 100  # Original selected IDs, including every version


In [ ]:
# Prevent a new checkout from being mislabeled while Python keeps old src modules.
import sys

if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError(
        "Restart runtime before setup; cached pipeline modules may belong to another revision."
    )

## Copy the filtered snapshot from Drive

Upload `arxiv-assigned-20000.jsonl` beside the full snapshot in your Drive input folder. Default: `MyDrive/arxiv-inputs/arxiv-assigned-20000.jsonl`; edit SNAPSHOT_DRIVE_PATH if your folder differs. This 39.99 MB file contains all 20,000 assigned IDs and 26,491 versions, retaining all original fields.

Run this cell to mount Drive and copy the filtered file to /content. The SHA-256 is checked before publishing the local file; a matching existing local file is reused. The Drive original is never modified or deleted. Crawl outputs and checkpoints are saved on Drive. No Kaggle download or Kaggle token is needed.

Use a fresh runtime so the full snapshot is absent from local disk. Copy/hash is setup time outside the benchmark; the filtered input is counted in benchmark disk measurements. For resume keep the same filtered bytes, path and run configuration.


In [ ]:
import hashlib
import json
import shutil
import tempfile
from pathlib import Path


def verify_snapshot(path):
    if not path.is_file():
        raise FileNotFoundError(f"Upload the filtered snapshot to: {path}")
    with path.open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    if actual != SNAPSHOT_SHA256:
        raise ValueError(f"Snapshot SHA-256 mismatch: {path}; check the input file")


if SNAPSHOT_PATH:
    snapshot_target = Path(SNAPSHOT_PATH)
    if snapshot_target.exists():
        verify_snapshot(snapshot_target)
        print("Reusing verified local snapshot:", snapshot_target)
    else:
        from google.colab import drive

        drive.mount("/content/drive")
        snapshot_source = Path(SNAPSHOT_DRIVE_PATH)
        if not snapshot_source.is_file():
            raise FileNotFoundError(
                f"Upload arxiv-assigned-20000.jsonl to {snapshot_source}; "
                "or edit SNAPSHOT_DRIVE_PATH to match its location."
            )
        snapshot_target.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(
            prefix="arxiv-input-", dir=snapshot_target.parent
        ) as temporary:
            copied = Path(temporary) / "snapshot.jsonl"
            shutil.copyfile(snapshot_source, copied)
            verify_snapshot(copied)
            copied.replace(snapshot_target)
        print("Copied and verified filtered snapshot:", snapshot_target)
    print("Snapshot bytes:", snapshot_target.stat().st_size)
else:
    print("Snapshot disabled; discovery will use the network fallback.")

## Authorize Google Drive

Use the Google account with access to the team folder. Drive may already be mounted for the input snapshot.


In [ ]:
import os
import subprocess
import sys
import uuid
from pathlib import Path

from google.colab import userdata

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path(DRIVE_DIR)
if not DRIVE.is_dir():
    raise FileNotFoundError("Create/check the intended team Drive folder before running")


## Resolve saved revision

For resume, read the saved Drive manifest and keep the same run ID and configuration.


In [ ]:
revision = CODE_REF
if RESUME:
    manifest_path = DRIVE / "runs" / RUN_ID / "MANIFEST.json"
    saved = json.loads(manifest_path.read_text())
    revision = saved["code_revision"]


## Check out the implementation

The crawler is pinned to the immutable revision shown below. Restart the runtime before changing revisions.


In [ ]:
repo_dir = Path("/content/CodeRepo")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", revision], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Implementation revision:", REVISION)

## Install dependencies

Package installation is setup and is outside the benchmark session.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True
)
if not Path("/usr/bin/time").is_file():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "time"], check=True)

## Optional API credentials and local workspace

ARXIV_CONTACT_EMAIL and S2_API_KEY are read from Colab Secrets if present. No values are printed.


In [ ]:
for name in ("ARXIV_CONTACT_EMAIL", "S2_API_KEY"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value.strip()
    print(name, "configured" if value else "not configured")
LOCAL_ROOT = Path("/content/m1-local") / uuid.uuid4().hex
if SNAPSHOT_PATH and not Path(SNAPSHOT_PATH).is_file():
    raise FileNotFoundError("Copy the filtered snapshot to SNAPSHOT_PATH before benchmarking")

## Preflight and exact scope

Defaults: Dang offset 0/count 2500, IDs 2604.20535–2604.23034. Use a new RUN_ID. For an initial smoke use COUNT=100 with a separate RUN_ID; use another fresh run for 2500. Use the existing team Drive folder and TEAM.json; initialize only if missing. The implementation must be published before Colab can check it out.


In [ ]:
from dataclasses import asdict

import psutil

from src.benchmark.checkpoints import prepare_drive
from src.benchmark.runner import BenchmarkConfig, plan_chunks, run_benchmark

if "archive_batch_size" not in BenchmarkConfig.__dataclass_fields__:
    raise RuntimeError("CODE_REF does not contain archive checkpoints; publish this implementation first")
prepare_drive(DRIVE, TEAM_ID, initialize=INITIALIZE_TEAM_FOLDER)
cfg = BenchmarkConfig(
    run_id=RUN_ID,
    mode=MODE,
    ranges=tuple(RANGES),
    offset=OFFSET,
    count=COUNT,
    chunk_size=CHUNK_SIZE,
    references=WITH_REFERENCES,
    snapshot=SNAPSHOT_PATH,
    storage_backend="drive",
    checkpoint_s=CHECKPOINT_S,
    archive_batch_size=ARCHIVE_BATCH_SIZE,
)
chunks = plan_chunks(cfg)

## Review scope and runtime

Confirm the CPU runtime, selected IDs and destination before starting. Use a fresh RUN_ID for a new experiment.


In [ ]:
print(asdict(cfg))
print("Selected IDs:", sum(len(c.ids) for c in chunks), "chunks:", len(chunks))
print("CPU count:", psutil.cpu_count(), "RAM MB:", psutil.virtual_memory().total / 1e6)
print("Storage: Drive", DRIVE)
subprocess.run(["lscpu"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)
selected_count = sum(len(chunk.ids) for chunk in chunks)
print("Expected archive slots:", (selected_count + ARCHIVE_BATCH_SIZE - 1) // ARCHIVE_BATCH_SIZE)
print("Drive layout: archives + index/log/telemetry; raw corpus remains local")


## Optional: prepare S3 source bundles on Drive

The hybrid uses S3 for snapshot-only v1 papers, then falls back to arXiv, then processes multi-version papers. S3 version is inferred from the snapshot and bundle timestamp; it is not independently verified. Selecting by manifest boundaries identifies candidates, not proof that a paper/version is present. Preparation is outside crawl session timing; retain the manifest and preparation logs. S3 requests and downloads may incur AWS Requester Pays charges.

### AWS credentials in Colab Secrets

1. In Colab, open the key icon **Secrets** in the left sidebar.
2. Add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`, each with its own value, and enable **Notebook access** for both.
3. Only when using temporary AWS credentials, also add `AWS_SESSION_TOKEN` and enable **Notebook access**. Long-term access keys do not have a session token.
4. Secrets are needed only if downloading bundles or fetching a manifest from S3. With the existing manifest and `S3_DOWNLOAD = False`, no S3 request is made.

Do not paste keys in notebook cells, output, GitHub, or chat. The AWS principal must be allowed to read the arXiv Requester Pays bucket. This notebook pins the preparation implementation through `CODE_REF`.

Existing bulk: this notebook defaults to `MyDrive/arxiv-s3/packages` and the saved manifest at `MyDrive/arxiv-s3/runs/20261102-135934/manifest.xml`. If the shared folder is elsewhere, add a shortcut to My Drive or edit these paths. With `S3_DOWNLOAD=False`, no AWS credentials or S3 request are needed.


In [ ]:
# This switch controls preparation only; it does not switch the crawl source.
PREPARE_S3 = True
S3_EXISTING_MANIFEST = "/content/drive/MyDrive/arxiv-s3/runs/20261102-135934/manifest.xml"
S3_BUNDLE_DIR = Path("/content/drive/MyDrive/arxiv-s3/packages")
S3_DOWNLOAD = False  # Reuse the existing manifest and bundles.
S3_BULK_DIR = Path("/content/drive/MyDrive/arxiv-nhom1/s3-bulk")
S3_MAX_DOWNLOAD_GB = 100.0  # hard bound for the total selected bundle sizes

RUN_CRAWL = False  # Enable after reviewing the plan and preparing bundles.

In [ ]:
if PREPARE_S3:
    if "chunks" not in globals() or "REVISION" not in globals():
        raise RuntimeError(
            "Run all setup cells through Review scope and runtime before preparing S3"
        )
    # Mount Drive for input bundles.
    from google.colab import drive

    drive.mount("/content/drive")
    import hashlib
    import importlib.util
    import json
    import time
    from datetime import UTC, datetime

    if importlib.util.find_spec("src.crawler.s3_prepare") is None:
        raise RuntimeError(
            "CODE_REF predates S3 preparation; publish and set it to the updated revision"
        )
    from src.crawler.s3_prepare import download_bundle, parse_manifest, select_bundles

    s3 = None
    if S3_DOWNLOAD or not S3_EXISTING_MANIFEST:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boto3"], check=True)
        import boto3

        access = userdata.get("AWS_ACCESS_KEY_ID")
        secret = userdata.get("AWS_SECRET_ACCESS_KEY")
        if not access or not secret:
            raise ValueError("Add AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY in Colab Secrets")
        try:
            session_token = userdata.get("AWS_SESSION_TOKEN")
        except Exception:
            session_token = None
        args = dict(aws_access_key_id=access, aws_secret_access_key=secret, region_name="us-east-1")
        if session_token:
            args["aws_session_token"] = session_token
        s3 = boto3.client("s3", **args)
        del access, secret, session_token, args
    if S3_EXISTING_MANIFEST:
        manifest_bytes = Path(S3_EXISTING_MANIFEST).read_bytes()
    else:
        response = s3.get_object(
            Bucket="arxiv", Key="src/arXiv_src_manifest.xml", RequestPayer="requester"
        )
        try:
            manifest_bytes = response["Body"].read()
        finally:
            response["Body"].close()
    manifest_sha = hashlib.sha256(manifest_bytes).hexdigest()
    all_ids = [item for chunk in chunks for item in chunk.ids]
    from src.crawler.s3_partition import partition_snapshot

    partition = partition_snapshot(Path(SNAPSHOT_PATH), all_ids)
    print(
        f"Snapshot groups: {len(partition.only_v1)} only-v1, "
        f"{len(partition.multi_version)} multi-version, "
        f"{len(partition.missing)} missing/invalid"
    )
    for name, group in (("only-v1", partition.only_v1), ("multi-version", partition.multi_version)):
        if group:
            print(f"{name}: {min(group)} .. {max(group)}")
    selected_ids = partition.only_v1
    bundles, uncovered = select_bundles(parse_manifest(manifest_bytes), selected_ids)
    total_bytes = sum(b.size for b in bundles)
    print(f"S3 manifest SHA-256: {manifest_sha}")
    print(
        f"Candidate bundles: {len(bundles)}, {total_bytes / 1e9:.2f} GB; "
        f"IDs outside boundaries: {len(uncovered)}"
    )
    if uncovered:
        print("IDs outside manifest boundaries:", uncovered)
    size_mismatches = []
    for bundle in bundles:
        existing = S3_BUNDLE_DIR / Path(bundle.key).name
        actual = existing.stat().st_size if existing.is_file() else None
        print(
            bundle.key,
            f"{bundle.first} .. {bundle.last}",
            "manifest_size",
            bundle.size,
            "drive_size",
            actual,
            "md5",
            bundle.md5,
        )
        if actual is not None and actual != bundle.size:
            size_mismatches.append((bundle.key, bundle.size, actual))
    if size_mismatches:
        print(
            f"NOTE: {len(size_mismatches)} bundle size(s) differ from manifest; "
            "MD5 will be checked before indexing."
        )
    if total_bytes / 1e9 > S3_MAX_DOWNLOAD_GB:
        raise ValueError("Selected bundle size exceeds S3_MAX_DOWNLOAD_GB; narrow the selection")
    S3_BULK_DIR.mkdir(parents=True, exist_ok=True)
    manifests = S3_BULK_DIR / "manifests"
    manifests.mkdir(exist_ok=True)
    manifest_path = manifests / f"{manifest_sha}.xml"
    if (
        manifest_path.exists()
        and hashlib.sha256(manifest_path.read_bytes()).hexdigest() != manifest_sha
    ):
        raise ValueError("Existing manifest file has unexpected content")
    if not manifest_path.exists():
        manifest_path.write_bytes(manifest_bytes)
    if S3_DOWNLOAD:
        run = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
        log_path = S3_BULK_DIR / f"preparation-{run}.jsonl"
        started = time.monotonic()
        with log_path.open("a", encoding="utf-8") as log_file:

            def record(event):
                log_file.write(json.dumps(event, ensure_ascii=False) + "\n")
                log_file.flush()

            record(
                {
                    "event": "plan",
                    "manifest_sha256": manifest_sha,
                    "bundle_count": len(bundles),
                    "bytes": total_bytes,
                    "uncovered_ids": uncovered,
                }
            )
            for number, bundle in enumerate(bundles, 1):
                print(f"[{number}/{len(bundles)}] {bundle.key}", flush=True)
                try:
                    event = download_bundle(s3, bundle, S3_BUNDLE_DIR)
                except Exception as exc:
                    record(
                        {
                            "event": "bundle_failed",
                            "key": bundle.key,
                            "error_type": type(exc).__name__,
                        }
                    )
                    raise
                record({"event": "bundle", **event, "manifest_sha256": manifest_sha})
                print(event["status"], f"{event['seconds']:.1f}s", flush=True)
            record({"event": "finished", "seconds": round(time.monotonic() - started, 3)})
        print("Preparation log:", log_path)
    else:
        print("Plan only; set S3_DOWNLOAD=True to download these bundles.")

## Crawl, archive checkpoint and benchmark

The original hybrid order is retained: all eligible S3 sources, then references + arXiv source fallback, then multi-version arXiv sources. Archive groups are fixed by the original selected ID order, not by phase or number of versions. A complete group includes source states and references for all its IDs. A final short group is allowed if COUNT is not divisible by 100.

Every checkpoint packages only completed-version bytes plus available metadata/references, and freezes work/log files into a separate archive. Partial snapshots may contain fewer than 100 completed papers; INDEX.json marks these complete=false. Packing, verification and upload are within checkpoint/session wall. Old immutable snapshots are retained on Drive; follow the latest INDEX.json rather than counting every archive file.

Local raw corpus is retained for QA and resume verification; archive staging lives under work/tmp and is in the sampler's disk scope. Sampled peaks can miss short spikes. Archive diagnostics split packing/verification from upload/verification; neither should be added again to wall time. Final evidence publication after sampler completion is outside session wall. A hard-killed session still has incomplete timing evidence.


In [ ]:
if RUN_CRAWL:
    from dataclasses import replace

    if not PREPARE_S3:
        raise ValueError("Run the S3 preparation cell first")
    missing_bundles = [b.key for b in bundles if not (S3_BUNDLE_DIR / Path(b.key).name).is_file()]
    if missing_bundles:
        print(f"WARNING: missing bundles will use arXiv fallback: {missing_bundles}")
    cfg = replace(
        cfg,
        source_mode="arxiv_s3",
        s3_manifest=str(manifest_path),
        s3_bundle_dir=str(S3_BUNDLE_DIR),
    )
    result = run_benchmark(
        cfg,
        drive_root=DRIVE,
        local_root=LOCAL_ROOT,
        revision=REVISION,
        team_id=TEAM_ID,
        resume=RESUME,
    )
    print(json.dumps(result, indent=2))
else:
    print("Crawl skipped: RUN_CRAWL=False")

## Inspect completion and measurements

Corpus coverage and timing completeness are separate. Session RSS includes the notebook/runner baseline and descendants; memory is sampled, so peaks can be missed. Session sampler wall includes crawl, retries and final corpus sync, but excludes initial manifest hash, input setup and final evidence publication. Discovery includes index preparation; do not add overlapping checkpoint or reference times to wall. Mean wall per required ID differs from per-paper lifecycle latency. Preserve logs and measurements even if finalization fails.


In [ ]:
import pandas as pd
from IPython.display import display

local = LOCAL_ROOT / RUN_ID
for name in ("coverage.json", "timing_audit.json"):
    p = local / "work" / name
    if p.exists():
        print(name, p.read_text())
audit = local / "book_csv/measurement_audit.json"
if audit.exists():
    print("Measurement audit:", audit.read_text())
errors = local / "work/cleanup_errors.json"
if errors.exists():
    print("Cleanup diagnostics:", errors.read_text())
for name in ("benchmark_summary.csv", "corpus_summary.csv"):
    p = local / "book_csv" / name
    if p.exists():
        print(name)
        display(pd.read_csv(p))
p = local / "book_csv/paper_attempt_timings.csv"
if p.exists():
    print("Per-paper attempts (first 10; all rows in ZIP)")
    display(pd.read_csv(p).head(10))
remote_index = DRIVE / "runs" / RUN_ID / "INDEX.json"
if remote_index.exists():
    index = json.loads(remote_index.read_text())
    batches = index.get("batches", [])
    print("Archive batches complete:", sum(b["complete"] for b in batches), "/", len(batches))
    print("Checkpointed versions:", index.get("versions"))
    display(pd.DataFrame([{k: b[k] for k in ("number", "papers", "versions", "complete", "bytes", "path")} for b in batches]))


## Export benchmark evidence

The downloadable evidence ZIP contains work, reports and logs. Corpus payloads are tar.gz archives on Drive, referenced by INDEX.json. The archive backend also publishes final work/book CSV evidence as a separate work archive after the sampler ends.


In [ ]:
from google.colab import files

from src.benchmark.runner import export_evidence, export_rescue

local = LOCAL_ROOT / RUN_ID
try:
    archive = export_evidence(cfg, local)
except Exception as exc:
    print("Statistics export failed; creating raw local rescue:", exc)
    archive = export_rescue(local)
print("Evidence:", archive)
files.download(str(archive))

## Rescue local data after an error

If benchmark/finalization fails, run this cell before resetting the runtime. It packages local data, logs and measurements without contacting the output storage.


In [ ]:
from google.colab import files as rescue_files

from src.benchmark.runner import export_rescue as make_rescue

archive = make_rescue(LOCAL_ROOT / RUN_ID)
print("Raw local rescue:", archive)
rescue_files.download(str(archive))

## Resume in a fresh runtime

Keep RUN_ID, config, snapshot bytes/path, ARCHIVE_BATCH_SIZE and Drive destination unchanged; set RESUME=True. The saved code revision is used. Restore verifies archive and member hashes, recreates raw local corpus (including empty version directories), then overlays independent telemetry and events. Already-complete chunks pass QA and are skipped. Current restore reads the complete selected checkpoint; this is not a lazy/instant resume optimization.

INDEX.json uses schema batch-archive-v1. Do not mix this run with the raw checkpoint backend or old run IDs. Old archive objects and generation indices remain for recovery; no automatic garbage collection is performed. A same-runtime Drive readback cannot guarantee cloud durability after a sudden runtime loss. Test fresh-runtime restore before the large run.
